In [1]:
!pip -q install transformers accelerate sentencepiece gradio


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [4]:
model_id = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

model = model.to(device)

print("Model loaded successfully!")


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


In [5]:
def answer_question(question, max_new_tokens=150):
    if not question or not question.strip():
        return "Please enter a question."

    prompt = (
        "Answer the following question clearly and accurately. "
        "If the question is asking for an explanation, explain it in simple language. "
        "Question: " + question
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


In [6]:
question = "What is machine learning?"

answer = answer_question(question)

print("Question:", question)
print("\nAnswer:", answer)


Question: What is machine learning?

Answer: Machine learning is a process that learns information from a computer.


In [7]:
question = input("Enter your question: ")

answer = answer_question(question)

print("\nQuestion:", question)
print("\nAnswer:", answer)


Enter your question: what is machine learning

Question: what is machine learning

Answer: A machine learning algorithm is a computer program that learns information from a computer program. A machine learning algorithm is a computer program that learns information from a computer program. A machine learning algorithm is an algorithm that learns information from a computer program.


In [8]:
def qa_interface(question, max_new_tokens):
    return answer_question(
        question,
        max_new_tokens=max_new_tokens
    )

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# AI Question–Answer System")
    gr.Markdown(
        "Enter a question below and the pre-trained FLAN-T5 model will generate an answer."
    )

    with gr.Row():

        with gr.Column():

            question_box = gr.Textbox(
                label="Enter Your Question",
                placeholder="Example: What is artificial intelligence?",
                lines=4
            )

            max_tokens = gr.Slider(
                minimum=30,
                maximum=300,
                value=150,
                step=10,
                label="Maximum Answer Length"
            )

            with gr.Row():
                ask_btn = gr.Button("Ask Question")
                clear_btn = gr.ClearButton()

        with gr.Column():

            answer_box = gr.Textbox(
                label="Generated Answer",
                lines=12
            )

    ask_btn.click(
        fn=qa_interface,
        inputs=[question_box, max_tokens],
        outputs=answer_box
    )

    clear_btn.add(
        [question_box, answer_box]
    )

demo.launch()


/tmp/ipykernel_5535/4292604109.py:7: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb1b3e5c3ffa99ea56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
